# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate all record sets, for each their fields, and for each field, its `@id` (unique identifier).

In [ ]:
record_set_objs = getattr(metadata, 'recordSet', [])

if not record_set_objs:
    # Attempt to infer record sets in case metadata.recordSet is empty
    print("No explicit record sets found in metadata. Attempting to detect record sets.")
    # mlcroissant might auto-detect from csv/tabular distribution
    # Let's try to enumerate possible record sets extracted by mlcroissant
    try:
        # This method returns generators for all available record sets
        detected_record_sets = dataset.list_record_sets()
        print("Record sets detected:")
        for rs in detected_record_sets:
            print(f" - @id: {rs['@id']} | name: {rs.get('name', '<no name>')}")
    except Exception as e:
        print(f"Unable to detect record sets: {e}")
else:
    print("Record sets defined in metadata:")
    for rs in record_set_objs:
        rs_id = getattr(rs, '@id', '<no id>')
        print(f"- RecordSet @id: {rs_id}")
        fields = getattr(rs, 'field', [])
        if fields:
            print("    Fields:")
            for field in fields:
                field_id = getattr(field, '@id', '<no id>')
                print(f"      - Field @id: {field_id}  name: {getattr(field, 'name', '<no name>')}")
        else:
            print("    No fields discoverable in this RecordSet.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's try to load data from the main record set associated with the dataset. If no record sets are explicitly listed in the metadata, we'll attempt to use the detected record sets (using `mlcroissant`) or the default record set id inferred from the Croissant file.

In [ ]:
# Try autodetection (as many Croissant datasets have only one main RecordSet)
try:
    record_sets = [rs['@id'] for rs in dataset.list_record_sets()]
except Exception:
    record_sets = []

# If list_record_sets not available or gives no result (older versions)
if not record_sets:
    # Try common known record set IDs used in Croissant
    # As a fallback, try '@id' from main distribution
    distributions = getattr(metadata, 'distribution', [])
    if distributions:
        # Sometimes distribution has contentUrl or similar, not always a record set
        possible_ids = [getattr(d, '@id', None) for d in distributions]
        record_sets = [rid for rid in possible_ids if rid]
if not record_sets:
    raise ValueError("No record sets found in the dataset.")

print("Record sets available for extraction:")
for i, rs_id in enumerate(record_sets):
    print(f"  {i+1}. {rs_id}")

dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        print(f"Record set '{record_set}' loaded: shape = {df.shape}")
        dataframes[record_set] = df
    except Exception as e:
        print(f"Could not load records for {record_set}: {e}")

# Pick the first non-empty DataFrame as main for analysis
main_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break
if main_record_set_id is None:
    raise ValueError("No data available in any record set.")
print(f"\nAnalysis will proceed using record set: {main_record_set_id}\nColumns:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare for further analysis.

Below, we automatically select a numeric field (column) if available, otherwise you can edit to choose a relevant field. We also show grouping by a categorical field such as `description` or similar.


In [ ]:
df = dataframes[main_record_set_id]

# Try to automatically find a numeric column to analyze
import numpy as np
numeric_candidate = None
for col in df.columns:
    # Try conversion to numeric
    s = pd.to_numeric(df[col], errors='coerce')
    # Accept the field if ~30% or more values can be converted
    valid = s.notna().sum()
    if valid > 0 and valid >= int(0.3 * len(df)):
        numeric_candidate = col
        print(f"Selected numeric field: {col}")
        # (There may be many, pick the first one.)
        break

if numeric_candidate is None:
    raise ValueError("No numeric field found for EDA. Edit this cell to select one manually.")

# Work with numeric column
numeric_field = numeric_candidate
s_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = s_numeric.mean()  # Use mean as example threshold
filtered_df = df[s_numeric > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): Total = {filtered_df.shape[0]}")

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (s_numeric[filtered_df.index] - s_numeric.mean()) / s_numeric.std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a grouping field
group_field = None
for col in df.columns:
    if col.lower() in ['description', 'group', 'category', 'sample']:
        group_field = col
        break
# If not found, pick another with few unique values but not numeric
if group_field is None:
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < 10 and col != numeric_field:
            group_field = col
            break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nMean {numeric_field} grouped by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the selected numeric field and, if available, the mean by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped, show horizontal bar of mean per group
if group_field is not None:
    plt.figure(figsize=(8,4))
    grouped_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    sns.barplot(x=grouped_means.values, y=grouped_means.index, palette="mako")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the dataset metadata and tabular data using mlcroissant.
- Identified main record set and its available fields using their `@id`s.
- Performed basic EDA with filtering, normalization, and grouping on a numeric field.
- Visualized the numeric field's distribution and group-wise mean values.

_For further analysis, adjust field selections (by their `@id`/column names) and develop custom filters or visualizations using the loaded DataFrames._